# KernelBench L2 12_Gemm_Multiply_LeakyReLU
#### Payload type: GEMM followed by elementwise operators, repeated forward passes

Source: [KernelBench/level2/12_Gemm_Multiply_LeakyReLU.py](https://github.com/ScalingIntelligence/KernelBench/blob/main/KernelBench/level2/12_Gemm_Multiply_LeakyReLU.py)

In [ ]:
%load_ext jumper_extension
%perfmonitor_fast_setup

In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Problem sizes and input factories, verbatim from KernelBench
batch_size = 1024
in_features  = 8192  
out_features = 8192
multiplier = 2.0
negative_slope = 0.1

def get_inputs():
    return [torch.rand(batch_size, in_features)]

def get_init_inputs():
    return [in_features, out_features, multiplier, negative_slope]

In [ ]:
torch.manual_seed(0)
inputs = [tensor.to(device) for tensor in get_inputs()]
iterations = 9000

In [ ]:
class Model(nn.Module):
    """
    Simple model that performs a Gemm, multiplies the result, and applies LeakyReLU.
    """
    def __init__(self, in_features, out_features, multiplier, negative_slope):
        super(Model, self).__init__()
        self.gemm = nn.Linear(in_features, out_features)
        self.multiplier = multiplier
        self.leaky_relu = nn.LeakyReLU(negative_slope)

    def forward(self, x):
        x = self.gemm(x)
        x = x * self.multiplier
        x = self.leaky_relu(x)
        return x


torch.manual_seed(0)
model = Model(*get_init_inputs()).to(device)
with torch.no_grad():
    for _ in range(iterations):
        output = model(*inputs)
    torch.cuda.synchronize()

In [ ]:
%perfmonitor_ai_review --benchmark --replay-mode full